In [1]:
import pandas as pd
from multicor_fa import mcfa_model

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

### Load in Intersim data

In [4]:
cluster_df = pd.read_csv(
    '../../sim_data/clustering_assignments.tsv', sep='\t', index_col=1
)
cluster_df.head()

,subjects,cluster.id
subject1,1,2
subject2,2,1
subject3,3,5
subject4,4,3
subject5,5,2


In [5]:
exp_data = pd.read_csv('../../sim_data/expression_data.tsv', sep='\t', index_col=0)
methyl_data = pd.read_csv('../../sim_data/methylation_data.tsv', sep='\t', index_col=0)
protein_data = pd.read_csv('../../sim_data/protein_data.tsv', sep='\t', index_col=0)

# Limit to first 1000 samples for now
exp_data = exp_data.iloc[:1000,:]
methyl_data = methyl_data.iloc[:1000,:]
protein_data = protein_data.iloc[:1000,:]
cluster_df = cluster_df.iloc[:1000]

In [6]:
# Create data dictionary
Y = {
    'exp': exp_data, 
    'methyl': methyl_data,
    'prot': protein_data
}

### Fitting model to complete data

In [7]:
for name, df in Y.items():
    print(f"Missing values in {name}:", df.isna().sum().sum())

Missing values in exp: 0
Missing values in methyl: 0
Missing values in prot: 0


In [8]:
print(cluster_df.shape, Y['exp'].shape, Y['methyl'].shape, Y['prot'].shape)

(1000, 2) (1000, 131) (1000, 367) (1000, 160)


In [9]:
%%time
mcfa_res_full = mcfa_model.fit(Y, missing_modes='raise')

Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.1972929239273071.
Fitting the model.
iter: 0 Likelihood: 29821.105760421276
Iter: 1 Likelihood: 29097.54813567292 Percent change: 0.024866618361609995 Time (s): 0.00032591819763183594
Iter: 2 Likelihood: 29006.733209858747 Percent change: 0.0031308222527902177 Time (s): 0.0006089210510253906
Iter: 3 Likelihood: 28982.97358877576 Percent change: 0.000819778585182453 Time (s): 0.0008699893951416016
Iter: 4 Likelihood: 28974.306920661005 Percent change: 0.00029911563160029894 Time (s): 0.001127004623413086
Iter: 5 Likelihood: 28970.037121435904 Percent change: 0.00014738673641332973 Time (s): 0.0013887882232666016
Iter: 6 Likelihood: 28967.188647686675 Percent change: 9.833449092604625e-05 Time (s): 0.0016448497772216797
Iter: 7 Likelihood: 28964.933582268906 Percent change: 7.785501773596148e-05 Time (s): 0.001894950866699

In [10]:
print(
    (mcfa_res_full.W['exp'] @ mcfa_res_full.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.801 -0.064  0.392 -0.213  0.279]
 [-0.064  0.826  0.035 -0.409 -0.381]
 [ 0.392  0.035  0.699 -0.31   0.241]
 [-0.213 -0.409 -0.31   0.65   0.248]
 [ 0.279 -0.381  0.241  0.248  0.546]]


In [11]:
print(
    (mcfa_res_full.L['exp'] @ mcfa_res_full.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [12]:
print(mcfa_res_full.Phi['exp'].round(3))

       0      1      2      3      4      5
0  0.341  0.022 -0.014 -0.005 -0.011 -0.028
1  0.022  0.360  0.013  0.005 -0.038  0.001
2 -0.014  0.013  0.295  0.005 -0.018 -0.030
3 -0.005  0.005  0.005  0.346 -0.021  0.001
4 -0.011 -0.038 -0.018 -0.021  0.363 -0.010
5 -0.028  0.001 -0.030  0.001 -0.010  0.463


### Fitting model to missing data
Each sample missing maximum 1 mode

In [13]:
Y_miss = Y.copy()
Y_miss['exp'] = Y_miss['exp'].iloc[20:]
Y_miss['methyl'] = Y_miss['methyl'].drop(
    index=Y_miss['methyl'].iloc[100:125].index.tolist()
)
Y_miss['prot'] = Y_miss['prot'].drop(
    index=Y_miss['prot'].iloc[500:530].index.tolist()
)

In [14]:
print(cluster_df.shape, Y_miss['exp'].shape, 
      Y_miss['methyl'].shape, Y_miss['prot'].shape)

(1000, 2) (980, 131) (975, 367) (970, 160)


In [15]:
Y['exp'].iloc[15:25, :5]

,ACACA,ACVRL1,AKT1,AKT1S1,ANXA1
subject16,1.239820,1.403926,1.274152,1.615542,1.600042
subject17,0.922452,1.710182,-0.152408,1.469768,2.594138
subject18,-1.026042,0.795188,-0.641408,1.119790,0.905765
subject19,1.873460,1.067464,1.546875,1.908176,1.775778
subject20,1.370386,2.953019,0.185173,0.597633,-0.316857
subject21,1.384600,1.176288,1.172802,1.428453,1.674897
subject22,-0.183378,3.641473,1.145361,0.492304,4.148478
subject23,1.382303,1.795173,0.649608,0.529566,2.785904
subject24,1.073222,3.237707,2.491712,-0.767261,1.677067
subject25,0.681191,1.592903,0.257474,1.673340,3.120252


####  Raise error for missing modes

In [16]:
%%time
# Confirm that missing_modes = raise works
mcfa_res_miss_raise = mcfa_model.fit(Y_miss, missing_modes='raise')

ValueError: Missing modes detected for some samples.

#### Impute mean

In [17]:
mcfa_res_miss_mean = mcfa_model.fit(Y_miss, missing_modes='impute_mean')

Missing modes detected for some samples, Imputing with the mean
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.1990463733673096.
Fitting the model.
iter: 0 Likelihood: 36969.18813576231
Iter: 1 Likelihood: 36427.8905981005 Percent change: 0.01485942580737952 Time (s): 0.00030803680419921875
Iter: 2 Likelihood: 36386.859660412934 Percent change: 0.0011276306356331764 Time (s): 0.0005779266357421875
Iter: 3 Likelihood: 36381.17224584261 Percent change: 0.00015632851332809502 Time (s): 0.0008320808410644531
Iter: 4 Likelihood: 36379.41091307724 Percent change: 4.841564833404083e-05 Time (s): 0.0010709762573242188
Iter: 5 Likelihood: 36378.465375257365 Percent change: 2.5991690691807976e-05 Time (s): 0.0013079643249511719
Iter: 6 Likelihood: 36377.83612752619 Percent change: 1.729755802315351e-05 Time (s): 0.0015430450439453125
Iter: 7 Likelihood: 36377.382988083016 Percent change: 1.245662568207524e-05 Time (s): 0.0017910003662109375
Iter: 8 Likelihood: 36377.04510701173 Percent change: 9.28830448694624e-06 Time (s): 0.0020220279693603516
Iter

In [18]:
print(
    (mcfa_res_miss_mean.W['exp'] @ mcfa_res_miss_mean.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.799 -0.068  0.397 -0.223  0.281]
 [-0.068  0.815  0.048 -0.408 -0.373]
 [ 0.397  0.048  0.69  -0.317  0.233]
 [-0.223 -0.408 -0.317  0.65   0.244]
 [ 0.281 -0.373  0.233  0.244  0.539]]


In [19]:
print(
    (mcfa_res_miss_mean.L['exp'] @ mcfa_res_miss_mean.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [20]:
print(mcfa_res_miss_mean.Phi['exp'].round(3))

       0      1      2      3      4      5
0  0.812 -0.067 -0.013  0.143 -0.076  0.016
1 -0.067  0.451 -0.003 -0.062  0.081  0.002
2 -0.013 -0.003  0.523  0.096 -0.067 -0.017
3  0.143 -0.062  0.096  0.666 -0.212 -0.025
4 -0.076  0.081 -0.067 -0.212  0.568 -0.040
5  0.016  0.002 -0.017 -0.025 -0.040  0.548


#### Drop samples with missing data

In [21]:
mcfa_res_miss_drop = mcfa_model.fit(Y_miss, missing_modes='drop')

Missing modes detected for some samples, dropping sampleswith missing modes. There are 925 samples remaining.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.2059963941574097.
Fitting the model.
iter: 0 Likelihood: 27592.75104490256
Iter: 1 Likelihood: 26924.338111388388 Percent change: 0.024825603167992077 Time (s): 0.0002892017364501953
Iter: 2 Likelihood: 26840.19408552931 Percent change: 0.0031350006483165855 Time (s): 0.0005502700805664062
Iter: 3 Likelihood: 26818.06986688629 Percent change: 0.0008249743084732144 Time (s): 0.00081634521484375
Iter: 4 Likelihood: 26809.986973990333 Percent change: 0.0003014881321575399 Time (s): 0.001062154769897461
Iter: 5 Likelihood: 26806.016720093805 Percent change: 0.0001481105506269547 Time (s): 0.0013072490692138672
Iter: 6 Likelihood: 26803.38044479827 Percent change: 9.835607493483976e-05 Time (s): 0.0015411376953125
Ite

In [22]:
print(
    (mcfa_res_miss_drop.W['exp'] @ mcfa_res_miss_drop.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.807 -0.063  0.4   -0.223  0.277]
 [-0.063  0.828  0.05  -0.399 -0.367]
 [ 0.4    0.05   0.696 -0.325  0.233]
 [-0.223 -0.399 -0.325  0.652  0.239]
 [ 0.277 -0.367  0.233  0.239  0.542]]


In [23]:
print(
    (mcfa_res_miss_drop.L['exp'] @ mcfa_res_miss_drop.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [24]:
print(mcfa_res_miss_drop.Phi['exp'].round(3))

       0      1      2      3      4      5
0  0.339  0.023  0.015 -0.001  0.006  0.027
1  0.023  0.364 -0.007 -0.002  0.046 -0.004
2  0.015 -0.007  0.295 -0.001 -0.019 -0.031
3 -0.001 -0.002 -0.001  0.336  0.022  0.007
4  0.006  0.046 -0.019  0.022  0.368 -0.011
5  0.027 -0.004 -0.031  0.007 -0.011  0.463


### Impute model

In [25]:
%%time
mcfa_res_miss_impute = mcfa_model.fit(Y_miss, missing_modes='impute_model')

Missing modes detected in input, they will be imputed duringmodel fitting.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.2023662328720093.
Fitting the model.
iter: 0 Likelihood: 38349.40622680884
Iter: 1 Likelihood: 37161.07849375623 Percent change: 0.03197775148674096 Time (s): 0.0006859302520751953
Iter: 2 Likelihood: 37898.92344643566 Percent change: -0.019468757568332166 Time (s): 0.0019159317016601562
Calculating feature importance.
CPU times: user 1.87 s, sys: 1.02 s, total: 2.89 s
Wall time: 579 ms


In [26]:
print(
    (mcfa_res_miss_impute.W['exp'] @ mcfa_res_miss_impute.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.832 -0.071  0.405 -0.224  0.288]
 [-0.071  0.865  0.038 -0.421 -0.401]
 [ 0.405  0.038  0.73  -0.329  0.251]
 [-0.224 -0.421 -0.329  0.677  0.254]
 [ 0.288 -0.401  0.251  0.254  0.572]]


In [27]:
print(
    (mcfa_res_miss_impute.L['exp'] @ mcfa_res_miss_impute.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [28]:
print(mcfa_res_miss_impute.Phi['exp'].round(3))

       0      1      2      3      4      5
0  0.548  0.044  0.014  0.028 -0.009  0.014
1  0.044  0.420  0.009 -0.028  0.042 -0.017
2  0.014  0.009  0.389  0.017 -0.025 -0.043
3  0.028 -0.028  0.017  0.381 -0.057 -0.000
4 -0.009  0.042 -0.025 -0.057  0.386  0.008
5  0.014 -0.017 -0.043 -0.000  0.008  0.416


Each sample missing up to 2 modes

In [29]:
Y_miss_2 = Y.copy()
Y_miss_2['exp'] = Y_miss_2['exp'].drop(
    index=Y_miss_2['exp'].iloc[100:175].index.tolist()
)
Y_miss_2['methyl'] = Y_miss_2['methyl'].drop(
    index=Y_miss_2['methyl'].iloc[150:200].index.tolist()
)
Y_miss_2['prot'] = Y_miss_2['prot'].drop(
    index=Y_miss_2['prot'].iloc[190:225].index.tolist()
)

In [30]:
%%time
mcfa_res_miss_impute_2 = mcfa_model.fit(Y_miss_2, missing_modes='impute_model')

Missing modes detected in input, they will be imputed duringmodel fitting.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.2139308452606201.
Fitting the model.
iter: 0 Likelihood: 40672.35379489782
Iter: 1 Likelihood: 39427.88900918834 Percent change: 0.03156305896619194 Time (s): 0.001508951187133789
Iter: 2 Likelihood: 40302.85646255063 Percent change: -0.021709812409334193 Time (s): 0.0023620128631591797
Calculating feature importance.
CPU times: user 1.82 s, sys: 1.01 s, total: 2.83 s
Wall time: 572 ms


In [31]:
print(
    (mcfa_res_miss_impute_2.W['exp'] @ mcfa_res_miss_impute_2.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.834 -0.064  0.412 -0.231  0.285]
 [-0.064  0.863  0.035 -0.422 -0.393]
 [ 0.412  0.035  0.727 -0.335  0.242]
 [-0.231 -0.422 -0.335  0.682  0.255]
 [ 0.285 -0.393  0.242  0.255  0.568]]


In [32]:
print(
    (mcfa_res_miss_impute_2.L['exp'] @ mcfa_res_miss_impute_2.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [33]:
print(mcfa_res_miss_impute_2.Phi['exp'].round(3))

       0      1      2      3      4      5
0  0.542  0.014  0.006 -0.000  0.019 -0.027
1  0.014  0.416 -0.014 -0.001 -0.020  0.022
2  0.006 -0.014  0.506  0.060  0.066 -0.045
3 -0.000 -0.001  0.060  0.364  0.009  0.005
4  0.019 -0.020  0.066  0.009  0.331  0.001
5 -0.027  0.022 -0.045  0.005  0.001  0.341


Larger blocks of missing data

In [34]:
Y_miss_3 = Y.copy()
Y_miss_3['exp'] = Y_miss_3['exp'].iloc[150:]
Y_miss_3['methyl'] = Y_miss_3['methyl'].drop(
    index=Y_miss_3['methyl'].iloc[120:420].index.tolist()
)
Y_miss_3['prot'] = Y_miss_3['prot'].drop(
    index=Y_miss_3['prot'].iloc[350:550].index.tolist()
)

In [35]:
%%time
mcfa_res_miss_impute_3 = mcfa_model.fit(Y_miss_3, missing_modes='impute_model')

Missing modes detected in input, they will be imputed duringmodel fitting.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.2941317558288574.
Fitting the model.
iter: 0 Likelihood: 53723.123551776465
Iter: 1 Likelihood: 62893.62288130573 Percent change: -0.1458096848202884 Time (s): 0.0007739067077636719
Calculating feature importance.
CPU times: user 1.37 s, sys: 641 ms, total: 2.01 s
Wall time: 498 ms


In [36]:
print(
    (mcfa_res_miss_impute_3.W['exp'] @ mcfa_res_miss_impute_3.W['exp'].T).values[:5,:5].round(3)
)

[[ 0.908 -0.079  0.445 -0.264  0.298]
 [-0.079  0.928  0.038 -0.441 -0.43 ]
 [ 0.445  0.038  0.785 -0.374  0.262]
 [-0.264 -0.441 -0.374  0.726  0.274]
 [ 0.298 -0.43   0.262  0.274  0.623]]


In [37]:
print(
    (mcfa_res_miss_impute_3.L['exp'] @ mcfa_res_miss_impute_3.L['exp'].T).values[:5,:5].round(3)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [38]:
print(mcfa_res_miss_impute_3.Phi['exp'].round(3))

       0      1      2      3      4      5
0  3.153  0.020  0.001 -0.044 -0.001 -0.001
1  0.020  2.102 -0.010  0.036  0.040  0.003
2  0.001 -0.010  1.777 -0.047  0.012 -0.028
3 -0.044  0.036 -0.047  1.284  0.011  0.016
4 -0.001  0.040  0.012  0.011  1.121  0.014
5 -0.001  0.003 -0.028  0.016  0.014  0.823
